# Clase 173 — JAX + Flax: fundamentos

Cubrimos `jit`, `grad`, `vmap` y un MLP con Flax. Fallback en numpy con autograd manual si JAX no está disponible.

In [ ]:
import numpy as np, time
JAX_OK = False
try:
    import jax, jax.numpy as jnp
    JAX_OK = True
    print(f'JAX {jax.__version__} | devices: {jax.devices()}')
except Exception:
    print('JAX no disponible → fallback con numpy + grad manual')

## 1. Fallback: f(x) = x³, derivada manual 3x²

In [ ]:
def f_np(x): return x ** 3
def grad_f_np(x): return 3 * x ** 2

for x in [1.0, 2.0, 3.0]:
    print(f'  f({x}) = {f_np(x):.2f}  |  f\'({x}) manual = {grad_f_np(x):.2f}')

## 2. JAX: `jit`, `grad`, `vmap`

In [ ]:
if JAX_OK:
    def f(x): return x ** 3
    df = jax.grad(f)
    print('grad de x³ via jax.grad:')
    for x in [1.0, 2.0, 3.0]:
        print(f'  jax.grad(f)({x}) = {df(x):.2f}  (esperado: {3*x*x:.2f})')

    # vmap: vectorizar grad sobre un batch
    xs = jnp.array([1.0, 2.0, 3.0, 4.0])
    print(f'\nvmap(grad(f))(xs) = {jax.vmap(df)(xs)}')
else:
    print('skip (JAX requerido)')

## 3. Benchmark JIT vs eager

In [ ]:
if JAX_OK:
    def slow_fn(x):
        for _ in range(50):
            x = jnp.tanh(x @ x.T) @ x
        return x.sum()
    fast_fn = jax.jit(slow_fn)
    key = jax.random.PRNGKey(42)
    X = jax.random.normal(key, (128, 64))
    # warmup
    _ = slow_fn(X).block_until_ready(); _ = fast_fn(X).block_until_ready()
    t0 = time.perf_counter()
    for _ in range(5): slow_fn(X).block_until_ready()
    t_eager = (time.perf_counter() - t0) / 5 * 1000
    t0 = time.perf_counter()
    for _ in range(5): fast_fn(X).block_until_ready()
    t_jit = (time.perf_counter() - t0) / 5 * 1000
    print(f'eager: {t_eager:.2f} ms/run | jit: {t_jit:.2f} ms/run | speedup: {t_eager/t_jit:.2f}x')
else:
    # Fallback numpy
    def slow_np(X):
        for _ in range(50): X = np.tanh(X @ X.T) @ X
        return X.sum()
    X = np.random.default_rng(42).standard_normal((128, 64))
    t0 = time.perf_counter(); [slow_np(X) for _ in range(3)]
    print(f'numpy slow_fn: {(time.perf_counter()-t0)/3*1000:.2f} ms/run')
    print('(JAX JIT daría ~5-20× speedup; en GPU/TPU mucho más)')

## 4. PRNG explícito

JAX no usa estado global. Cada llamada aleatoria necesita una **key**, y splitéas la key para no re-usar.

In [ ]:
if JAX_OK:
    key = jax.random.PRNGKey(42)
    k1, k2, k3 = jax.random.split(key, 3)
    print('sample 1:', jax.random.normal(k1, (3,)))
    print('sample 2:', jax.random.normal(k2, (3,)))
    print('sample 3:', jax.random.normal(k3, (3,)))
    print('mismo key → mismo sample:', jax.random.normal(k1, (3,)))
else:
    print('numpy.random.default_rng(42) — estado implícito, no functional')

## 5. Flax conceptual: MLP

Flax separa **definición del módulo** de **parámetros** (PyTrees). Init explícito con una key, apply explícito con (params, x).

In [ ]:
try:
    import flax.linen as nn
    class MLP(nn.Module):
        @nn.compact
        def __call__(self, x):
            x = nn.Dense(32)(x); x = nn.relu(x)
            x = nn.Dense(16)(x); x = nn.relu(x)
            return nn.Dense(1)(x)
    model = MLP()
    key = jax.random.PRNGKey(42)
    params = model.init(key, jnp.ones((1, 8)))
    out = model.apply(params, jnp.ones((4, 8)))
    print('Flax MLP output shape:', out.shape)
    print('params tree:', jax.tree_util.tree_map(lambda x: x.shape, params))
except Exception as e:
    print(f'Flax no disponible: {type(e).__name__}')
    print('Definición conceptual:')
    print('  class MLP(nn.Module):')
    print('      @nn.compact')
    print('      def __call__(self, x):')
    print('          x = nn.relu(nn.Dense(32)(x))')
    print('          return nn.Dense(1)(x)')
    print('  params = model.init(key, dummy_x)')
    print('  y = model.apply(params, x)')

## 6. Por qué JAX vs PyTorch

| | JAX | PyTorch |
|--|--|--|
| Paradigma | functional, transforms | imperativo, mutable |
| JIT | XLA via `jax.jit` | torch.compile (más nuevo) |
| Autograd | `jax.grad` (pura) | `.backward()` (stateful) |
| TPU | first-class | vía torch_xla |
| Ecosistema | Flax, Haiku, Equinox | enorme |
| Sweet spot | research, científico, TPU | producción, comunidad |

## Ejercicio guiado

1. Implementar `value_and_grad` para una loss MSE y entrenar el MLP con `optax`.
2. Vectorizar con `pmap` sobre múltiples devices (si hay).
3. Comparar `torch.compile(slow_fn)` vs `jax.jit(slow_fn)` en CPU.

## Conclusiones

- JAX = numpy + transforms (jit, grad, vmap, pmap) — composable y puro.
- PRNG explícito hace todo reproducible y paralelizable.
- Flax es el go-to para deep learning en JAX (research, Gemini, AlphaFold).